In [ ]:
import pickle
import cv2
import numpy as np
import dlib
import requests
import os
import matplotlib.pyplot as plt

# ==========================================
# 2. CONFIG & PATH
# ==========================================
model_path = './model/face_classifier.pkl'
predictor_path = './predictor/shape_predictor_68_face_landmarks.dat'
face_rec_model_path = './predictor/dlib_face_recognition_resnet_model_v1.dat'

# URL gambar contoh
image_url = 'https://ichef.bbci.co.uk/ace/standard/976/cpsprodpb/6EDE/production/_132928382_gettyimages-2071909768.jpg'
temp_filename = "test_image.jpg"

# Pastikan model dlib tersedia
required_files = [predictor_path, face_rec_model_path]
for f in required_files:
    if not os.path.exists(f):
        print(f"[WARNING] Dlib file not found at path: {f}")

# ==========================================
# 3. CLASS & FUNCTIONS
# ==========================================

class FaceFeatureExtractor:
    def __init__(self, predictor_path, face_rec_model_path):
        self.detector = dlib.get_frontal_face_detector()
        self.shape_predictor = dlib.shape_predictor(predictor_path)
        self.face_rec_model = dlib.face_recognition_model_v1(face_rec_model_path)

    def extract_features(self, image):
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        faces = self.detector(gray, 1)
        features = []
        locations = []

        for face in faces:
            shape = self.shape_predictor(gray, face)
            face_descriptor = np.array(self.face_rec_model.compute_face_descriptor(image, shape))
            features.append(face_descriptor)
            locations.append((face.left(), face.top(), face.right(), face.bottom()))

        return features, locations

def download_image_safe(url, filename):
    print(f"[INFO] Attempting to download image...")
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Referer': 'https://www.google.com/',
        'Accept': 'image/avif,image/webp,image/apng,image/svg+xml,image/*,*/*;q=0.8'
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=15)
        if response.status_code == 200:
            with open(filename, 'wb') as f:
                f.write(response.content)
            print("[INFO] Download successful via URL.")
            return True
        else:
            print(f"[ERROR] Download failed. Status Code: {response.status_code}")
            return False
    except Exception as e:
        print(f"[ERROR] Exception during download: {e}")
        return False

def classify_and_visualize(image_path, extractor, classifier):
    image = cv2.imread(image_path)
    if image is None:
        print(f"[ERROR] Failed to read image from path: {image_path}")
        return None

    features, locations = extractor.extract_features(image)

    if len(features) == 0:
        print("[WARNING] No faces detected in the image.")
        return image

    print(f"[INFO] Found {len(features)} face(s).")

    for feature, (x1, y1, x2, y2) in zip(features, locations):
        prediction = classifier.predict([feature])[0]
        name = str(prediction)

        print(f"[RESULT] Prediction: {name}")

        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.rectangle(image, (x1, y2 - 30), (x2, y2), (0, 255, 0), cv2.FILLED)
        font = cv2.FONT_HERSHEY_DUPLEX
        cv2.putText(image, name, (x1 + 6, y2 - 6), font, 0.6, (0, 0, 0), 1)

    return image

def show_image(image):
    """Menampilkan gambar menggunakan matplotlib (tanpa Colab)"""
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 8))
    plt.imshow(rgb_image)
    plt.axis('off')
    plt.show()

# ==========================================
# 4. MAIN PROGRAM
# ==========================================

# Load model klasifikasi
try:
    print(f"[INFO] Loading classification model...")
    with open(model_path, 'rb') as f:
        classifier = pickle.load(f)
except Exception as e:
    print(f"[ERROR] Failed to load Pickle model: {e}")
    classifier = None

# Inisialisasi extractor
try:
    extractor = FaceFeatureExtractor(predictor_path, face_rec_model_path)
except Exception as e:
    print(f"[ERROR] Failed to load dlib model: {e}")
    exit()

# MAIN LOGIC
process_success = False

# Coba unduh gambar dari URL
if download_image_safe(image_url, temp_filename):
    process_success = True
else:
    # Jika gagal, minta user masukkan path file lokal
    print("\n[INPUT REQUIRED] Download failed. Please enter the local path to your image file:")
    user_input = input("Image path (e.g., ./my_photo.jpg): ").strip()
    if os.path.isfile(user_input):
        temp_filename = user_input
        process_success = True
        print(f"[INFO] Using local file: {temp_filename}")
    else:
        print(f"[ERROR] File not found: {user_input}")

# Proses gambar jika berhasil
if process_success and classifier is not None:
    try:
        output_image = classify_and_visualize(temp_filename, extractor, classifier)
        if output_image is not None:
            print("\n--- DISPLAYING RESULT ---")
            show_image(output_image)
    except Exception as e:
        print(f"[ERROR] Failed to process image: {e}")
elif classifier is None:
    print("[STOP] Cannot process because the classification model (pickle) failed to load.")
else:
    print("[STOP] No image available to process.")